# Lab A — 用 `adk eval` 評保健品 agent（Colab · Service Account 版）

延續你蓋的**保健品文案小組**，這一關**回頭評它**：走對流程沒（trajectory）＋ 產出夠好沒（response）。

> **認證用 Service Account JSON**（不是 `auth.authenticate_user()`）：公司政策擋「第三方 notebook 用**你的 Google 帳號**存取 GCP」，
> 但 **SA 是「機器人身分」、不走你的帳號 OAuth → 繞過封鎖**，而且走**真的 Vertex**（Lab B 也共用同一份）。**Runtime → Run all**。

## 0. 安裝（google-adk 的 `[eval]` 提供 adk eval）

In [ ]:
!pip install -q "google-adk[eval]==1.37.0"

## 1. 上傳 Service Account JSON（講師提供）

執行這格 → 跳出上傳鈕 → 選講師給的 `*.json`。
> ⚠️ 這是憑證，別外流、別 commit 進 git；課後講師會停用該 SA。

In [ ]:
from google.colab import files
import os, json

print("請上傳 Service Account JSON 檔案（講師提供）：")
uploaded = files.upload()
sa_filename = os.path.abspath(list(uploaded.keys())[0])   # 絕對路徑：cd 進 repo 後才找得到
with open(sa_filename) as f:
    sa_info = json.load(f)

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = sa_filename
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = sa_info["project_id"]
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
print("✅ Vertex 認證完成，project =", sa_info["project_id"])

## 2. 拿 agent 程式碼（clone repo）＋寫 .env

In [ ]:
!git clone -q https://github.com/amelielee-tech/adk-workshop.git

# adk eval 讀 repo 根目錄 .env → 寫入 Vertex 設定（用剛上傳的 SA，路徑用絕對路徑）
with open("adk-workshop/.env", "w") as f:
    f.write(
        f"GOOGLE_APPLICATION_CREDENTIALS={os.environ['GOOGLE_APPLICATION_CREDENTIALS']}\n"
        f"GOOGLE_GENAI_USE_VERTEXAI=TRUE\n"
        f"GOOGLE_CLOUD_PROJECT={os.environ['GOOGLE_CLOUD_PROJECT']}\n"
        f"GOOGLE_CLOUD_LOCATION=global\n"
    )
print("repo cloned；.env 寫好（Vertex + SA）")
!ls adk-workshop/lab_eval/

## 3. 跑 `adk eval`（評 lab2_multi_agent）

`adk eval` **〈評誰〉〈用哪份考卷〉** `--config`**〈及格線〉**。會真的把 agent 跑兩次，約 1–2 分鐘。

In [ ]:
!cd adk-workshop && adk eval lab2_multi_agent lab_eval/copy_agent.evalset.json \
  --config_file_path lab_eval/test_config.json 2>&1 | grep -E "Tests passed|Tests failed|Using evaluation" 

## 4. 讀成乾淨的分數表（兩軸分開看）

In [ ]:
import json, glob
import pandas as pd

rows = []
for f in sorted(glob.glob("adk-workshop/lab2_multi_agent/.adk/eval_history/*.json")):
    d = json.load(open(f))
    for c in d["eval_case_results"]:
        r = {"case": c["eval_id"]}
        for m in c["overall_eval_metric_results"]:
            r[m["metric_name"]] = round(m["score"], 3)
            r[m["metric_name"] + " → "] = "PASS" if m["eval_status"] == 1 else "FAIL"
        rows.append(r)

pd.DataFrame(rows)

**怎麼讀**：
- `tool_trajectory_avg_score`＝走對流程沒。門檻 1.0、用 IN_ORDER。
- `response_match_score`＝文案字面（ROUGE-1 F1）像不像。**中文創作型天生低又飄**（門檻只 0.15）——這正是下一關 Lab B 改用 LLM-judge 的理由。

## 5. 效率：打一次 agent 看 latency + token

In [ ]:
!cd adk-workshop && python lab_eval/probe_efficiency.py

## 收尾

- **agent 評估分兩軸**：走對流程（trajectory）＋ 產出夠好（response）。
- **每個指標都有極限**：ROUGE 對創作型中文很弱 → 所以 Lab B 用 LLM-judge。
- **效率**（latency/token）只有真的跑 agent 才量得到。

一句話：**eval 讓「感覺不錯」變成「量得出來」。**